# Синтетическая балансировка: влияние на качество

Проверяем честно, дает ли порождение синтетических примеров рецидивов прирост качества. Модель - лучшая не-CatBoost из отложенного теста, случайный лес на наборе no_collinear с тюнингованными гиперпараметрами (ноутбук 07). CatBoost не берем: он работает с категориями нативно, а методы семейства SMOTE требуют числовой кодировки.

Стратегии: без балансировки (none), взвешивание классов (class_weight), SMOTENC (корректный метод для смешанных данных), BorderlineSMOTE, ADASYN и обычный SMOTE поверх one-hot (некорректен для категорий, оставлен для контраста).

Честность оценки обеспечивается так. Оверсэмплинг выполняется только на train внутри фолдов, оценка всегда на реальной, не синтетической, отложенной части фолда. Считаем пять seed-ов (разные разбиения и инициализация) и приводим среднее и стандартное отклонение, чтобы отличить реальный эффект от случайности. Смотрим не только на ROC-AUC: синтетика двигает баланс классов и порог, а не ранжирование, и часто ухудшает калибровку, поэтому отдельно следим за PR-AUC и Brier. Логика в `src/synthetic.py`.

In [ ]:
import sys, pathlib, warnings
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))
warnings.filterwarnings('ignore')

import pandas as pd
from IPython.display import display
from src import synthetic

table = synthetic.run()
print()
display(table)

## Вывод

Синтетическая балансировка не улучшила качество. По ранжированию (ROC-AUC) и по PR-AUC, важной при дисбалансе, ни один синтетический метод не превзошел простые варианты: ROC-AUC у none 0.729 и class_weight 0.737 против 0.715-0.725 у SMOTENC, BorderlineSMOTE и ADASYN, причем различия лежат внутри разброса по seed-ам (SD около 0.02). PR-AUC у синтетики даже чуть ниже (0.515-0.530 против 0.559 у none).

Калибровка от синтетики ухудшается, как и ожидалось: Brier растет с 0.186-0.187 (none, class_weight) до 0.193-0.200 у синтетических методов. Искусственные точки искажают распределение и делают вероятности менее честными.

Рост чувствительности при пороге 0.5 (с 0.24 у none до 0.45-0.58 у синтетики) - это не улучшение модели, а сдвиг рабочей точки: синтетика меняет априорный баланс классов и тем самым опускает эффективный порог. Тот же сдвиг чувствительности мы получаем напрямую, выбирая порог под целевую чувствительность (ноутбук 09), без искажения данных и без потери калибровки.

Итог: при текущих данных синтетическая балансировка не добавляет информации, только перераспределяет рабочую точку и портит калибровку. В пайплайн ее не включаем, баланс чувствительности и специфичности задаем порогом. На более крупных или сильнее несбалансированных данных вопрос можно пересмотреть.